#  Performance study: NSR vs. V magnitude

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker

# PlatoSim
import platosim.plot            as pt
import platosim.utilities       as ut
import platosim.referenceFrames as rf
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

## Download data from FTP

In [ ]:
# User parameters
idir  = "/lhome/nicholas/data/platosimPaper/NSR"
cfile = idir + "/starcat_all_SPF_CamVis24_NewCat_targets.ftr"

## Test example of how the NSR is calculated

In [ ]:
# Load light curve object for the first star only and unpack the data
lcs = LightCurve(f"{idir}/000000001", mode="multi")
lcs.unpack()

In [ ]:
# Fetch all files
filenames = lcs.files("hdf5")
filenames

In [ ]:
# Merge all N-CAM light curves
lc, ncam, flag = lcs.merge(quarter=1, flux_group_mean=True, suffix='hdf5')

In [ ]:
# Calculate the NSR in 1h
lc.getNSR(influx='ppm')

In [ ]:
# Show plot to verify result
lc.plot(time_unit="h", flux_unit='ppm', ncam=ncam, quarter=1, binsize=1, alpha=0.5, figsize=(9,5));

## Analysis of Merged N-CAM LCs

In [ ]:
ofile = idir + "/results_per_star.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perStar(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df = pd.read_feather(ofile)
df

In [ ]:
# Load input catalogue
dc = pd.read_feather(cfile)
N = len(dc)

# Merge the two data frames
mag  = np.array([])
ncon = np.array([])
for i in range(N):
    nobs = len(df[df.star == i+1])    
    mags = dc.mag.iloc[i] * np.ones(nobs)
    mag  = np.concatenate((mag, mags))
    ncons = dc.ncon.iloc[i] * np.ones(nobs)
    ncon  = np.concatenate((ncon, ncons))                
df["mag"]  = mag
df["ncon"] = ncon
df

In [ ]:
df['mag'] = mag[:len(df)]

In [ ]:
# dx[dx.star == 1000]
df

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncam", Vmag=True, residuals="multi", 
                                legend=True, grid=False, figsize=(12,8)) #(9,8))

# Set axes limits
ax.set_xlim(6, 12.77)
ax.set_ylim(5, 1000) #340)

# We change the fontsize of minor ticks label 
ticks_minor = [5, 6, 20, 30, 40, 50, 60, 70, 80, 200, 300]
ax.yaxis.set_minor_locator(mticker.FixedLocator(ticks_minor))
ax.set_yticklabels(ticks_minor)
ticks_major = [1, 10, 100]
ax.yaxis.set_major_locator(mticker.FixedLocator(ticks_major))
ax.set_yticklabels(ticks_major)

# Save figure
fig.savefig('NSRvsV.png', bbox_inches='tight', dpi=300);

## Analysis of Indiviual N-CAM LCs

In [ ]:
ofile = idir + "/results_per_camera.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perCamera(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df = pd.read_feather(ofile)
df

In [ ]:
# Load input catalogue
dc = pd.read_feather(cfile)
N = len(dc)

# Merge the two data frames
mag  = np.array([])
ncon = np.array([])
for i in range(N):
    nobs = len(df[df.star == i+1])    
    mags = dc.mag.iloc[i] * np.ones(nobs)
    mag  = np.concatenate((mag, mags))
    ncons = dc.ncon.iloc[i] * np.ones(nobs)
    ncon  = np.concatenate((ncon, ncons))                
df["mag"]  = mag
df["ncon"] = ncon
df

In [ ]:
mag = df.mag.unique()
mag

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncon", Vmag=True, residuals="multi", 
                                legend=True, grid=False, figsize=(9,8))

# Set axes limits
# ax.set_xlim(7, 12.77)
# ax.set_ylim(6, 340)

#### From average of N-CAMs

In [ ]:
dx = pd.DataFrame()
for i in range(1, 10001):
    star = df.loc[df["star"] == i]
    ncam = len(star)
    if ncam > 0:
        mag  = star.mag.iloc[0]
        nsr  = star.NSR.mean() / np.sqrt(ncam)
        data = {'star':i, "mag":mag, "ncam":ncam, "NSR":nsr}
        dx = dx.append(data, ignore_index=True)

In [ ]:
dx

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(dx, column="ncam", Vmag=True, residuals="multi", 
                                legend=True, grid=False, figsize=(9,8))

# Set axes limits
# ax.set_xlim(7, 12.77)
# ax.set_ylim(6, 340)

# We change the fontsize of minor ticks label 
ticks_minor = [5, 6, 20, 30, 40, 50, 60, 70, 80, 200, 300]
ax.yaxis.set_minor_locator(mticker.FixedLocator(ticks_minor))
ax.set_yticklabels(ticks_minor)
ticks_major = [1, 10, 100]
ax.yaxis.set_major_locator(mticker.FixedLocator(ticks_major))
ax.set_yticklabels(ticks_major);

# Save figure
# fig.savefig('NSRvsV.png', bbox_inches='tight', dpi=300);